# MoonStyle • Fooocus GPU Launcher
این نوت‌بوک موتور Fooocus را روی GPU کولب اجرا می‌کند و بعد یک دکمه برای برگشت و اتصال خودکار به اپ MoonStyle می‌دهد.

**کار شما:** فقط سلول زیر را اجرا کنید و صبر کنید تا دکمه سبز «برگشت به MoonStyle و اتصال» ظاهر شود.

> اگر GPU فعال نیست: Runtime → Change runtime type → T4 GPU

In [ ]:
import os, re, time, subprocess, urllib.parse, shutil, sys
from pathlib import Path
from IPython.display import display, HTML

print('MoonStyle: در حال آماده‌سازی Fooocus روی GPU...')

# Clean stale processes from a previous run inside the same runtime.
subprocess.run(['pkill', '-f', 'Fooocus-API/main.py'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['pkill', '-f', 'cloudflared tunnel'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

os.chdir('/content')
repo = Path('/content/Fooocus-API')
venv = Path('/content/fooocus-venv')

if not repo.exists():
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/mrhan1993/Fooocus-API.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
if not venv.exists():
    subprocess.run(['uv', 'venv', '--python', '3.11', '--seed', str(venv)], check=True)

py = str(venv / 'bin' / 'python')
pip = str(venv / 'bin' / 'pip')
marker = Path('/content/.moonstyle_fooocus_requirements_ok')
if not marker.exists():
    print('MoonStyle: نصب وابستگی‌ها؛ بار اول چند دقیقه طول می‌کشد...')
    subprocess.run([pip, 'install', '-r', str(repo / 'requirements.txt')], check=True)
    marker.write_text('ok')

os.chdir(repo)
api_log_path = Path('/content/moonstyle-fooocus-api.log')
api_log = open(api_log_path, 'w')
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['PYTHONUNBUFFERED'] = '1'

cmd = [
    py, 'main.py',
    '--host', '0.0.0.0',
    '--port', '8888',
    '--always-high-vram',
    '--all-in-fp16'
]
api_proc = subprocess.Popen(cmd, stdout=api_log, stderr=subprocess.STDOUT, env=env)

import requests
ready = False
for i in range(360):
    if api_proc.poll() is not None:
        break
    try:
        r = requests.get('http://127.0.0.1:8888/ping', timeout=2)
        if r.ok and 'pong' in r.text.lower():
            ready = True
            break
    except Exception:
        pass
    if i % 12 == 0:
        print(f'MoonStyle: Fooocus در حال راه‌اندازی است... {i*5} ثانیه')
    time.sleep(5)

if not ready:
    api_log.flush()
    tail = api_log_path.read_text(errors='ignore')[-7000:] if api_log_path.exists() else ''
    print(tail)
    raise RuntimeError('Fooocus API بالا نیامد. آخرین لاگ بالا نمایش داده شد.')

print('MoonStyle: Fooocus آماده شد. در حال ساخت لینک امن موقت...')
cf = Path('/content/cloudflared')
if not cf.exists():
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', str(cf)], check=True)
    os.chmod(cf, 0o755)

cf_log_path = Path('/content/moonstyle-cloudflared.log')
cf_log = open(cf_log_path, 'w')
cf_proc = subprocess.Popen(
    [str(cf), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8888'],
    stdout=cf_log, stderr=subprocess.STDOUT
)

public_url = None
for _ in range(90):
    time.sleep(1)
    if cf_proc.poll() is not None:
        break
    cf_log.flush()
    txt = cf_log_path.read_text(errors='ignore') if cf_log_path.exists() else ''
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    cf_log.flush()
    print(cf_log_path.read_text(errors='ignore')[-4000:] if cf_log_path.exists() else '')
    raise RuntimeError('لینک عمومی ساخته نشد. سلول را یک بار دیگر اجرا کنید.')

deep = 'moonstyle://fooocus?url=' + urllib.parse.quote(public_url, safe='')
display(HTML(f'''
<div dir="rtl" style="font-family:Arial,sans-serif;border:1px solid #e5e7eb;border-radius:18px;padding:20px;background:#fff;max-width:680px">
  <div style="font-size:22px;font-weight:700;margin-bottom:8px">✅ Fooocus آماده است</div>
  <div style="color:#666;margin-bottom:12px">تا وقتی این Runtime روشن است، موتور تصویر MoonStyle کار می‌کند.</div>
  <div style="direction:ltr;text-align:left;background:#f5f5f5;border-radius:10px;padding:10px;word-break:break-all;margin-bottom:14px">{public_url}</div>
  <a href="{deep}" style="display:inline-block;background:#10A37F;color:white;text-decoration:none;padding:13px 18px;border-radius:13px;font-weight:700">برگشت به MoonStyle و اتصال</a>
  <div style="font-size:12px;color:#777;margin-top:12px">اگر اندروید پرسید لینک با چه برنامه‌ای باز شود، MoonStyle را انتخاب کنید.</div>
</div>
'''))
print('API URL:', public_url)
print('برای خاموش کردن موتور، Runtime را Disconnect کنید.')
